# 02 · Patient networks for: long hospital stays

Question: **will this hospital stay last 7 days or more?** (asked at the end of the first day)

We build three networks out of the same hospital stays:

| network | each stay is described by | why |
|---|---|---|
| **honest** | diagnoses from *before* the prediction time | the one you could really use |
| **cheating** | the patient's *whole* record, future included | shows what peeking does |
| **random** | the honest network with its links shuffled (10 times) | what "no real pattern" looks like |

Two stays get a link if each is in the other's top-10 most similar stays (similarity = which diagnoses they share, with rare diagnoses counting more).

In [ ]:
import matplotlib.pyplot as plt

from ehrgraph.eval.aggregate import write_aggregate
from ehrgraph.netstats.task_analysis import analyze_task, friendly_table, task_graphs_from_labels
from ehrgraph.project import setup
from ehrgraph.viz import network_figures as nf

p = setup()  # finds the data in data/omop/


nf.apply_style()
TASK = "guo_los"

## Build the networks and measure everything

This takes a couple of minutes. It saves one small summary file (no patient data) for the report.

In [ ]:
labels = p.labels(TASK)
nets = task_graphs_from_labels(p.diagnoses, labels, TASK)
results = analyze_task(nets)
write_aggregate(results.rows, p.results / "network_stats_task_a.csv")
print(f"{len(nets.nodes):,} stays, {nets.prevalence:.1%} with the outcome")

## What the networks look like

Each dot is one stay: **purple = long stay**, grey = not long stay. Only the biggest connected piece of each network is drawn. If similar patients share outcomes, purple dots should bunch together in the honest and cheating pictures, but not in the random one.

In [ ]:
fig = nf.layout_figure(nets, "long stay", "not long stay")
nf.save(fig, p.figures / "02_network_pictures.png", dpi=80)
plt.show()

## Basic shape

How the real networks differ from random wiring. Every stay has the same number of links in "honest" and "random", so any difference comes from *who* is linked to whom.

In [ ]:
friendly_table(
    results.rows,
    {
        "n_edges": "links",
        "largest_cc_fraction": "share of stays in the biggest piece",
        "transitivity": "closed triangles (friends of friends are linked)",
        "lcc_avg_shortest_path_hops": "average hops between two stays",
        "leiden_modularity": "how cleanly it splits into groups (modularity)",
    },
)

## Do linked patients share outcomes?

This is the main question. **Homophily** measures how much more often linked stays share the outcome than chance would give (0 = no more than chance). One thing to watch: many links join two stays of the *same patient*, so we also check homophily without those.

In [ ]:
friendly_table(
    results.rows,
    {
        "class_insensitive_homophily": "homophily",
        "class_insensitive_homophily_excl_same_patient": "homophily, ignoring same-patient links",
        "same_patient_edge_fraction": "share of links between two stays of the same patient",
        "leiden_outcome_eta_squared": "how much of the outcome the groups explain",
    },
)

Group the stays into communities (Leiden), then look at how often each group has the outcome. Flat line = groups don't care about the outcome. Spread out = they do.

In [ ]:
fig = nf.community_outcome_figure(nets, results, "long stay")
nf.save(fig, p.figures / "02_group_outcome_rates.png")
plt.show()

## What does cheating actually change?

Left: links that only the honest network has, only the cheating one has, or both. How often does each kind join two stays with the same outcome? Right: homophily with and without same-patient links.

In [ ]:
fig = nf.leakage_edges_figure(nets, results)
nf.save(fig, p.figures / "02_what_cheating_changes.png")
plt.show()

And the single diagnosis that best separates the outcome *in the cheating version*. If one stands out a lot, the future record is basically giving the answer away.

In [ ]:
top = results.top_leaky_codes.head(5)
top.assign(diagnosis=top.code.map(p.names(top.code)))[
    ["diagnosis", "positive_rate", "negative_rate", "gap"]
].round(2)

## Being "central" vs the outcome

Quick check whether the most connected or most "in the middle" stays are more likely to have the outcome. 0.5 = no link at all.

In [ ]:
friendly_table(
    results.rows,
    {
        "degree_centrality_auroc_vs_label": "most links",
        "betweenness_centrality_auroc_vs_label": "most bridge-like",
        "closeness_centrality_auroc_vs_label": "closest to everyone",
    },
    conditions=("safe", "leaky"),
)

## What we learned

- **Real networks are clumpy, not random.** Closed triangles: 0.49 (honest) vs 0.001 (random). Paths are longer too: 9.4 vs 5.3 hops. Similar patients come in tight little groups.
- **Linked patients really do share outcomes.** Homophily is **0.21** for honest vs **0.01** for random. In plain words: about 40% of a long-stay patient's neighbours are also long-stay, when only 25% of all stays are.
- **The groups mean something.** Leiden groups range from almost no long stays to over 70%, and they explain about 11% of the outcome (random: 1%).
- **Cheating doesn't help here.** Homophily is 0.20 vs 0.21 honest. The links that only the cheating network has are slightly *worse* (68.6% same outcome vs 70.8%). Makes sense: "how long will this stay last" isn't a diagnosis that shows up later in the record.
- **Same-patient links:** about 38% of links join two stays of the same person. They actually *lower* homophily a bit (0.21 → 0.24 without them), because the same person can have one long stay and one short one.
- **Being central doesn't mean anything** for the outcome (all around 0.5).
- **The strongest future "giveaway"** is pleural effusion (63% vs 37%). That's a real signal, but nowhere near the answer.